# Goodreads Book Trends: Database Creation

## Overview

This notebook creates the final SQLite relational database for the Goodreads book trends project using the cleaned and validated datasets produced during data preparation.

The database is designed to organize Goodreads book information into related entity and relationship tables while also integrating a second dataset containing popular Science Fiction and Fantasy books.

## Database Structure

The database includes the following primary entity tables:

- **BOOKS** — Stores unique Goodreads book records and their metadata.
- **AUTHORS** — Stores unique authors.
- **GENRES** — Stores unique Goodreads genres.
- **SOURCE_GENRES** — Identifies the original Goodreads genre dataset from which a book was collected.
- **SCIFI_FANTASY_BOOKS** — Stores records from the separate Science Fiction and Fantasy dataset.

Relationship tables establish the many-to-many relationships between the entities:

- **BOOK_AUTHORS** — Connects books to authors.
- **BOOK_GENRES** — Connects books to Goodreads genres.
- **BOOK_SOURCE_GENRES** — Connects books to their original source genre datasets.
- **BOOK_DATASET_MATCHES** — Connects matching books between the primary Goodreads dataset and the separate Science Fiction/Fantasy dataset.

## Database Integrity

Primary keys, foreign keys, unique constraints, and composite keys are used to maintain the integrity of the relational structure. Foreign-key enforcement is enabled in SQLite, and integrity checks are performed after the tables and relationships are populated.

The completed database is saved as:

`../Data/goodreads_capstone.db`

## Cross-Dataset Integration

The database also provides a structured connection between the two Goodreads-derived datasets. Matching records are linked using the `BOOK_DATASET_MATCHES` table, allowing ratings, publication information, and other attributes to be compared across datasets.

The final sections of this notebook validate the database structure, row counts, relationships, foreign keys, and sample relational queries before closing the database connection.

### 1. Load Prepared Datasets

The finalized Parquet files created during the data preparation process are loaded into pandas DataFrames.

The primary relational tables include books, authors, genres, source genres, and their associated relationship tables. The prepared Science Fiction and Fantasy dataset and its cross-dataset matching table are also loaded.

These DataFrames provide the cleaned and validated data that will be used to construct the SQLite relational database.

In [ ]:
from pathlib import Path
import pandas as pd

prepared_folder = Path("../Data/prepared")

books = pd.read_parquet(prepared_folder / "books.parquet")
authors = pd.read_parquet(prepared_folder / "authors.parquet")
book_authors = pd.read_parquet(prepared_folder / "book_authors.parquet")
genres = pd.read_parquet(prepared_folder / "genres.parquet")
book_genres = pd.read_parquet(prepared_folder / "book_genres.parquet")
source_genres = pd.read_parquet(prepared_folder / "source_genres.parquet")
book_source_genres = pd.read_parquet(
    prepared_folder / "book_source_genres.parquet"
)

scifi_fantasy_books = pd.read_parquet(
    prepared_folder / "scifi_fantasy_books.parquet"
)

book_dataset_matches = pd.read_parquet(
    prepared_folder / "book_dataset_matches.parquet"
)


print(
    "Sci-Fi/Fantasy books:",
    scifi_fantasy_books.shape
)

print(
    "Cross-dataset matches:",
    book_dataset_matches.shape
)

Sci-Fi/Fantasy books: (2491, 11)
Cross-dataset matches: (1940, 2)


### 2. Validate Prepared Datasets

Before creating the SQLite database, the prepared DataFrames are summarized to confirm that the expected tables and relationships were loaded successfully.

The record counts for each entity and relationship table are displayed, providing a quick verification of the data available for database construction.

In [ ]:
print("=" * 60)
print("PREPARED DATASET SUMMARY")
print("=" * 60)

print(f"Books:                 {len(books):,}")
print(f"Authors:               {len(authors):,}")
print(f"Book-Author links:     {len(book_authors):,}")
print(f"Genres:                {len(genres):,}")
print(f"Book-Genre links:      {len(book_genres):,}")
print(f"Source Genres:         {len(source_genres):,}")
print(f"Book-Source links:     {len(book_source_genres):,}")

PREPARED DATASET SUMMARY
Books:                 1,487,805
Authors:               482,461
Book-Author links:     1,488,070
Genres:                1,433
Book-Genre links:      4,981,671
Source Genres:         100
Book-Source links:     4,291,574


### 3. Inspect DataFrame Structures

The data types of each prepared DataFrame are inspected before the tables are loaded into SQLite.

This check helps confirm that identifiers, text fields, ratings, and other attributes have appropriate data types and are ready to be mapped to the corresponding SQLite column types.

In [ ]:
print("\nBOOKS")
print(books.dtypes)

print("\nAUTHORS")
print(authors.dtypes)

print("\nBOOK_AUTHORS")
print(book_authors.dtypes)

print("\nGENRES")
print(genres.dtypes)

print("\nBOOK_GENRES")
print(book_genres.dtypes)

print("\nSOURCE_GENRES")
print(source_genres.dtypes)

print("\nBOOK_SOURCE_GENRES")
print(book_source_genres.dtypes)


BOOKS
book_id          int64
book_key           str
name               str
pub_year         int16
star_rating    float64
num_ratings      int64
isbn_clean         str
dtype: object

AUTHORS
author_id      int64
author_name      str
dtype: object

BOOK_AUTHORS
book_id      int64
author_id    int64
dtype: object

GENRES
genre_id      int64
genre_name      str
dtype: object

BOOK_GENRES
book_id     int64
genre_id    int64
dtype: object

SOURCE_GENRES
source_genre_id       int64
source_genre       category
dtype: object

BOOK_SOURCE_GENRES
book_id            int64
source_genre_id    int64
dtype: object


### 4. Preview Prepared Tables

The first few records from each prepared table are displayed to visually verify the data before database creation.

This provides a final inspection of the table contents and helps confirm that the primary entity tables and relationship tables contain the expected columns and values.

In [ ]:
display(books.head())
display(authors.head())
display(book_authors.head())
display(genres.head())
display(book_genres.head())
display(source_genres.head())
display(book_source_genres.head())

,book_id,book_key,name,pub_year,star_rating,num_ratings,isbn_clean
0,1,isbn:9780060932299,Sharpe's Devil,1992,4.14,8141,9780060932299
1,2,isbn:9780786836628,Blood Fever,2006,4.02,7516,9780786836628
2,3,isbn:9783770476374,Detektiv Conan vs. Kaito Kid,2004,4.33,231,9783770476374
3,4,isbn:9781772752014,Marvel's Captain America: Sub Rosa,2016,3.39,74,9781772752014
4,5,isbn:9780821714782,The Awakening,1984,3.87,305,9780821714782


,author_id,author_name
0,1,!
1,2,"""Albert"""
2,3,"""Big"" John McCarthy"
3,4,"""J"""
4,5,"""Janosch"""


,book_id,author_id
0,1,44881
1,2,71940
2,3,159536
3,4,100324
4,5,209437


,genre_id,genre_name
0,1,10th century
1,2,11th century
2,3,12th century
3,4,13th century
4,5,14th century


,book_id,genre_id
0,1,644
1,1,640
2,1,1370
3,1,36
4,1,878


,source_genre_id,source_genre
0,1,action
1,2,adult
2,3,adventure
3,4,amazon
4,5,american_history


,book_id,source_genre_id
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1


### 5. Create SQLite Database

A SQLite database is created to store the cleaned and prepared Goodreads data in a relational structure.

The database file is stored in the project's `Data` directory as `goodreads_capstone.db`. This database will contain the entity tables, relationship tables, and cross-dataset integration tables needed for the project's SQL analysis.

In [ ]:
import sqlite3

db_path = Path("../Data/goodreads_capstone.db")

print("Database location:")
print(db_path.resolve())

Database location:
C:\Users\sarah\Projects\Book_Genre_Trends_Analysis\Data\goodreads_capstone.db


### 6. Connect to SQLite Database

A connection is established to the SQLite database, and foreign-key enforcement is enabled.

Foreign-key enforcement ensures that relationships between tables remain valid. For example, a record in `BOOK_AUTHORS` cannot reference a book or author that does not exist in the corresponding parent table.

The connection status and foreign-key setting are displayed as a validation check.

In [ ]:
conn = sqlite3.connect(db_path)

conn.execute("PRAGMA foreign_keys = ON;")

print("Connected to SQLite database.")
print("Foreign keys enabled:", 
      conn.execute("PRAGMA foreign_keys;").fetchone()[0])

Connected to SQLite database.
Foreign keys enabled: 1


### 7. Create Database Tables

The SQLite database schema is created using SQL `CREATE TABLE` statements.

The database is structured around the primary `BOOKS`, `AUTHORS`, `GENRES`, and `SOURCE_GENRES` entity tables, along with relationship tables that implement the many-to-many relationships between books and authors, genres, and source datasets.

The `SCIFI_FANTASY_BOOKS` table stores the second Goodreads dataset and provides the source records used for cross-dataset analysis.

Primary keys, unique constraints, foreign keys, and cascading delete rules are defined to maintain referential integrity and prevent duplicate entity or relationship records.

In [ ]:
cursor = conn.cursor()

cursor.executescript("""
    
    
    -- BOOKS

    
    CREATE TABLE IF NOT EXISTS books (
        book_id INTEGER PRIMARY KEY,
        book_key TEXT NOT NULL UNIQUE,
        name TEXT NOT NULL,
        pub_year INTEGER,
        star_rating REAL,
        num_ratings INTEGER,
        isbn_clean TEXT
    );


    
    -- AUTHORS
    
    
    CREATE TABLE IF NOT EXISTS authors (
        author_id INTEGER PRIMARY KEY,
        author_name TEXT NOT NULL UNIQUE
    );


    
    -- GENRES
    
    
    CREATE TABLE IF NOT EXISTS genres (
        genre_id INTEGER PRIMARY KEY,
        genre_name TEXT NOT NULL UNIQUE
    );


   
    -- SOURCE GENRES
    
    
    CREATE TABLE IF NOT EXISTS source_genres (
        source_genre_id INTEGER PRIMARY KEY,
        source_genre TEXT NOT NULL UNIQUE
    );


    
    -- BOOK_AUTHORS
    
    CREATE TABLE IF NOT EXISTS book_authors (
        book_id INTEGER NOT NULL,
        author_id INTEGER NOT NULL,

        PRIMARY KEY (book_id, author_id),

        FOREIGN KEY (book_id)
            REFERENCES books(book_id)
            ON DELETE CASCADE,

        FOREIGN KEY (author_id)
            REFERENCES authors(author_id)
            ON DELETE CASCADE
    );


    
    -- BOOK_GENRES
    
    
    CREATE TABLE IF NOT EXISTS book_genres (
        book_id INTEGER NOT NULL,
        genre_id INTEGER NOT NULL,

        PRIMARY KEY (book_id, genre_id),

        FOREIGN KEY (book_id)
            REFERENCES books(book_id)
            ON DELETE CASCADE,

        FOREIGN KEY (genre_id)
            REFERENCES genres(genre_id)
            ON DELETE CASCADE
    );


    
    -- BOOK_SOURCE_GENRES
    
    
    CREATE TABLE IF NOT EXISTS book_source_genres (
        book_id INTEGER NOT NULL,
        source_genre_id INTEGER NOT NULL,

        PRIMARY KEY (book_id, source_genre_id),

        FOREIGN KEY (book_id)
            REFERENCES books(book_id)
            ON DELETE CASCADE,

        FOREIGN KEY (source_genre_id)
            REFERENCES source_genres(source_genre_id)
            ON DELETE CASCADE
    );

    -- Create Sci-Fi/Fantasy Books Table
    
          CREATE TABLE IF NOT EXISTS scifi_fantasy_books (
            scifi_fantasy_book_id INTEGER PRIMARY KEY,
            title TEXT,
            author TEXT,
            pub_year INTEGER,
            avg_rate REAL,
            num_rate INTEGER,
            shelved INTEGER,
            series_name TEXT,
            series_num REAL,
            source_genre TEXT
        );
""")


conn.commit()

print("All database tables created successfully.")

All database tables created successfully.


### 8. Create Cross-Dataset Relationship Table

A `BOOK_DATASET_MATCHES` relationship table is created to connect matching records between the primary Goodreads `BOOKS` table and the separate `SCIFI_FANTASY_BOOKS` dataset.

The table uses a composite primary key consisting of both book identifiers, preventing the same cross-dataset relationship from being stored more than once.

Foreign-key constraints ensure that every relationship references valid records in both datasets. This table provides the structural connection needed to compare Goodreads records with the Science Fiction and Fantasy dataset during SQL analysis.

In [ ]:
conn.execute(
    """
    CREATE TABLE IF NOT EXISTS book_dataset_matches (
        book_id TEXT NOT NULL,
        scifi_fantasy_book_id INTEGER NOT NULL,

        PRIMARY KEY (
            book_id,
            scifi_fantasy_book_id
        ),

        FOREIGN KEY (book_id)
            REFERENCES books(book_id),

        FOREIGN KEY (scifi_fantasy_book_id)
            REFERENCES scifi_fantasy_books(
                scifi_fantasy_book_id
            )
    );
    """
)

### 9. Recreate the Sci-Fi/Fantasy Books Table

The `SCIFI_FANTASY_BOOKS` table is recreated before loading the prepared Sci-Fi/Fantasy dataset.

The existing table is removed and replaced with a table generated directly from the prepared DataFrame. This ensures that the database contains the finalized Sci-Fi/Fantasy records, including the assigned `scifi_fantasy_book_id` values.

The number of inserted records is displayed to confirm that the dataset was loaded successfully.

In [ ]:
conn.execute("DROP TABLE IF EXISTS scifi_fantasy_books")
conn.commit()

scifi_fantasy_books.to_sql(
    "scifi_fantasy_books",
    conn,
    if_exists="replace",
    index=False
)

print(
    "Inserted",
    len(scifi_fantasy_books),
    "Sci-Fi/Fantasy books."
)

Inserted 2491 Sci-Fi/Fantasy books.


### 10. Verify Sci-Fi/Fantasy Table Schema

The structure of the `SCIFI_FANTASY_BOOKS` table is inspected using SQLite's `PRAGMA table_info` command.

This verifies that the recreated table contains the expected columns and data types, including the `scifi_fantasy_book_id` primary key and the book metadata required for cross-dataset analysis.

In [10]:
display(
    pd.read_sql_query(
        "PRAGMA table_info(scifi_fantasy_books)",
        conn
    )
)

,cid,name,type,notnull,dflt_value,pk
0,0,scifi_fantasy_book_id,INTEGER,0,None,0
1,1,title,TEXT,0,None,0
2,2,author,TEXT,0,None,0
3,3,pub_year,INTEGER,0,None,0
4,4,avg_rate,REAL,0,None,0
5,5,num_rate,INTEGER,0,None,0
6,6,shelved,INTEGER,0,None,0
7,7,series_name,TEXT,0,None,0
8,8,series_num,REAL,0,None,0
9,9,source_genre,TEXT,0,None,0


### 11. Insert Entity Tables

The prepared entity DataFrames are inserted into their corresponding SQLite tables.

The primary entity tables populated at this stage are `BOOKS`, `AUTHORS`, `GENRES`, and `SOURCE_GENRES`. The prepared `SCIFI_FANTASY_BOOKS` records are also loaded into the database.

The data is appended to the tables rather than replacing the existing schema. After insertion, the number of Sci-Fi/Fantasy records is reported and the transaction is committed to save the changes.

In [ ]:
books.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False
)

authors.to_sql(
    "authors",
    conn,
    if_exists="append",
    index=False
)

genres.to_sql(
    "genres",
    conn,
    if_exists="append",
    index=False
)

source_genres.to_sql(
    "source_genres",
    conn,
    if_exists="append",
    index=False
)

scifi_fantasy_books.to_sql(
    "scifi_fantasy_books",
    conn,
    if_exists="append",
    index=False
)

print(
    "Inserted",
    len(scifi_fantasy_books),
    "Sci-Fi/Fantasy books."
)
conn.commit()

print("Entity tables populated successfully.")

Inserted 2491 Sci-Fi/Fantasy books.
Entity tables populated successfully.


### 12. Insert Relationship Tables

The prepared relationship DataFrames are inserted into the corresponding SQLite relationship tables.

These tables establish the many-to-many relationships between:

- Books and authors through `BOOK_AUTHORS`
- Books and genres through `BOOK_GENRES`
- Books and their original source datasets through `BOOK_SOURCE_GENRES`

The transaction is committed after insertion to permanently save the relationship records in the database.

In [ ]:
book_authors.to_sql(
    "book_authors",
    conn,
    if_exists="append",
    index=False
)

book_genres.to_sql(
    "book_genres",
    conn,
    if_exists="append",
    index=False
)

book_source_genres.to_sql(
    "book_source_genres",
    conn,
    if_exists="append",
    index=False
)

conn.commit()

print("Relationship tables populated successfully.")

Relationship tables populated successfully.


### 13. Recreate the Cross-Dataset Relationship Table

The `BOOK_DATASET_MATCHES` table is recreated using the finalized cross-dataset relationship DataFrame.

The previous version of the table is removed to ensure that outdated or incorrectly structured relationships are not retained. The current validated relationships are then inserted into SQLite.

The number of cross-dataset relationships inserted is displayed, and the transaction is committed to save the updated table.

In [ ]:
conn.execute(
    "DROP TABLE IF EXISTS book_dataset_matches"
)

conn.commit()


book_dataset_matches.to_sql(
    "book_dataset_matches",
    conn,
    if_exists="replace",
    index=False
)


print(
    "Inserted",
    len(book_dataset_matches),
    "cross-dataset relationships."
)


conn.commit()

Inserted 1940 cross-dataset relationships.


### 14. Commit Database Changes

All database changes made during the table creation and data-loading process are committed to the SQLite database.

This ensures that the entity tables, relationship tables, Sci-Fi/Fantasy records, and cross-dataset relationships are permanently saved.

In [ ]:
conn.commit()

print("Database changes committed successfully.")

Database changes committed successfully.


### 15. Verify Database Tables

The SQLite database schema is queried to retrieve a list of all tables currently present in the database.

This provides a high-level verification that the expected entity, relationship, and cross-dataset tables were created successfully.

In [ ]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

display(tables)

,name
0,authors
1,book_authors
2,book_dataset_matches
3,book_genres
4,book_source_genres
5,books
6,genres
7,scifi_fantasy_books
8,source_genres


### 16. Validate Database Row Counts

The number of records in each primary entity and relationship table is counted directly from the SQLite database.

Comparing these counts with the prepared DataFrames provides a basic validation that the data was transferred into the database without unexpected record loss or duplication.

In [ ]:
tables = [
    "books",
    "authors",
    "book_authors",
    "genres",
    "book_genres",
    "source_genres",
    "book_source_genres"
]

print("=" * 60)
print("DATABASE ROW COUNTS")
print("=" * 60)

for table in tables:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table:25} {count:,}")

DATABASE ROW COUNTS
books                     1,487,805
authors                   482,461
book_authors              1,488,070
genres                    1,433
book_genres               4,981,671
source_genres             100
book_source_genres        4,291,574


### 17. Verify Integration Table Row Counts

The row counts of the `SCIFI_FANTASY_BOOKS` and `BOOK_DATASET_MATCHES` tables are verified directly against the SQLite database.

This confirms that the Sci-Fi/Fantasy dataset and the cross-dataset relationship records were successfully inserted and are available for subsequent SQL analysis.

In [ ]:
verification = pd.read_sql_query(
    """
    SELECT
        (SELECT COUNT(*)
         FROM scifi_fantasy_books)
            AS scifi_fantasy_books,

        (SELECT COUNT(*)
         FROM book_dataset_matches)
            AS book_dataset_matches;
    """,
    conn
)

display(verification)

,scifi_fantasy_books,book_dataset_matches
0,4982,1940


### 18. Test Cross-Dataset Relationship

A test query is used to verify that the `BOOK_DATASET_MATCHES` table correctly connects records from the primary Goodreads `BOOKS` table with the `SCIFI_FANTASY_BOOKS` table.

The query joins the three tables using their matching identifiers and returns a sample of matched records. Goodreads information such as title, publication year, rating, and number of ratings is displayed alongside the corresponding Sci-Fi/Fantasy information.

This confirms that the cross-dataset relationship can be successfully traversed through SQL and that the integrated data is ready for comparative analysis.

In [ ]:
cross_dataset_test = pd.read_sql_query(
    """
    SELECT
        b.book_key,
        b.name AS goodreads_title,
        b.pub_year AS goodreads_pub_year,
        b.star_rating AS goodreads_rating,
        b.num_ratings AS goodreads_num_ratings,

        sf.scifi_fantasy_book_id,
        sf.title AS scifi_fantasy_title,
        sf.author AS scifi_fantasy_author,
        sf.pub_year AS scifi_fantasy_pub_year,
        sf.source_genre,
        sf.avg_rate AS scifi_fantasy_rating,
        sf.num_rate AS scifi_fantasy_num_ratings

    FROM book_dataset_matches m

    JOIN books b
        ON m.book_key = b.book_key

    JOIN scifi_fantasy_books sf
        ON m.scifi_fantasy_book_id = sf.scifi_fantasy_book_id

    LIMIT 20;
    """,
    conn
)

display(cross_dataset_test)

,book_key,goodreads_title,goodreads_pub_year,goodreads_rating,goodreads_num_ratings,scifi_fantasy_book_id,scifi_fantasy_title,scifi_fantasy_author,scifi_fantasy_pub_year,source_genre,scifi_fantasy_rating,scifi_fantasy_num_ratings
0,title:harry potter and the chamber of secrets|...,Harry Potter and the Chamber of Secrets,1998,4.43,4500160,2,Harry Potter and the Chamber of Secrets,J.K. Rowling,1998,Fantasy,4.43,3409926
1,title:harry potter and the chamber of secrets|...,Harry Potter and the Chamber of Secrets,1998,4.43,4500160,2,Harry Potter and the Chamber of Secrets,J.K. Rowling,1998,Fantasy,4.43,3409926
2,isbn:9780439655484,Harry Potter and the Prisoner of Azkaban,1999,4.58,4849237,3,Harry Potter and the Prisoner of Azkaban,J.K. Rowling,1999,Fantasy,4.58,3595393
3,isbn:9780439655484,Harry Potter and the Prisoner of Azkaban,1999,4.58,4849237,3,Harry Potter and the Prisoner of Azkaban,J.K. Rowling,1999,Fantasy,4.58,3595393
4,title:harry potter and the goblet of fire|auth...,Harry Potter and the Goblet of Fire,2000,4.57,4198921,5,Harry Potter and the Goblet of Fire,J.K. Rowling,2000,Fantasy,4.57,3164528
5,title:harry potter and the goblet of fire|auth...,Harry Potter and the Goblet of Fire,2000,4.57,4198921,5,Harry Potter and the Goblet of Fire,J.K. Rowling,2000,Fantasy,4.57,3164528
6,title:harry potter and the half-blood prince|a...,Harry Potter and the Half-Blood Prince,2005,4.58,3661172,6,Harry Potter and the Half-Blood Prince,J.K. Rowling,2005,Fantasy,4.58,2923256
7,title:harry potter and the half-blood prince|a...,Harry Potter and the Half-Blood Prince,2005,4.58,3661172,6,Harry Potter and the Half-Blood Prince,J.K. Rowling,2005,Fantasy,4.58,2923256
8,isbn:9780439358064,Harry Potter and the Order of the Phoenix,2003,4.50,3799643,7,Harry Potter and the Order of the Phoenix,J.K. Rowling,2003,Fantasy,4.50,3019296
9,isbn:9780439358064,Harry Potter and the Order of the Phoenix,2003,4.50,3799643,7,Harry Potter and the Order of the Phoenix,J.K. Rowling,2003,Fantasy,4.50,3019296


### 19. Summarize Cross-Dataset Matches

The matched books are summarized by their original Sci-Fi/Fantasy source genre.

For each source genre, the query calculates the number of matched books and compares the average Goodreads rating with the average rating from the Sci-Fi/Fantasy dataset.

This provides an initial validation of the cross-dataset integration while also demonstrating how the relational database can be used to compare information from multiple source datasets.

In [ ]:
cross_dataset_summary = pd.read_sql_query(
    """
    SELECT
        sf.source_genre,
        COUNT(DISTINCT m.scifi_fantasy_book_id) AS matched_books,
        ROUND(AVG(b.star_rating), 2) AS avg_goodreads_rating,
        ROUND(AVG(sf.avg_rate), 2) AS avg_scifi_fantasy_rating
    FROM book_dataset_matches m

    JOIN books b
        ON m.book_key = b.book_key

    JOIN scifi_fantasy_books sf
        ON m.scifi_fantasy_book_id =
           sf.scifi_fantasy_book_id

    GROUP BY sf.source_genre
    ORDER BY matched_books DESC;
    """,
    conn
)

display(cross_dataset_summary)

,source_genre,matched_books,avg_goodreads_rating,avg_scifi_fantasy_rating
0,Science Fiction,970,3.98,3.97
1,Fantasy,933,4.11,4.10


### 20. Validate Foreign-Key Integrity

SQLite's foreign-key integrity check is performed to verify that all foreign-key relationships in the database reference valid records.

A successful result confirms that the relational structure does not contain orphaned records or broken foreign-key references. If any violations are detected, the specific issues are displayed for further investigation.

In [ ]:
foreign_key_check = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()

if len(foreign_key_check) == 0:
    print("Foreign-key integrity check PASSED.")
else:
    print("Foreign-key integrity check FAILED.")
    print(foreign_key_check)

Foreign-key integrity check PASSED.


### 21. Inspect Database Schema

The SQLite schema is inspected by listing all tables currently stored in the database.

This final schema check confirms that the database contains the expected entity tables, relationship tables, and cross-dataset integration tables before proceeding to SQL analysis.

In [ ]:
schema = pd.read_sql_query("""
    SELECT
        name,
        type
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

display(schema)

,name,type
0,authors,table
1,book_authors,table
2,book_dataset_matches,table
3,book_genres,table
4,book_source_genres,table
5,books,table
6,genres,table
7,scifi_fantasy_books,table
8,source_genres,table


### 22. Inspect Table Columns and Constraints

The columns and basic structural properties of each database table are inspected using SQLite's `PRAGMA table_info` command.

This verifies the column names, data types, nullability, and primary-key definitions across the database tables. The results provide a final structural check before beginning the SQL analysis phase.

In [ ]:
for table in tables:
    print(f"\n{'=' * 60}")
    print(table.upper())
    print("=" * 60)

    table_info = pd.read_sql_query(
        f"PRAGMA table_info({table});",
        conn
    )

    display(table_info)


BOOKS


,cid,name,type,notnull,dflt_value,pk
0,0,book_id,INTEGER,0,None,1
1,1,book_key,TEXT,1,None,0
2,2,name,TEXT,1,None,0
3,3,pub_year,INTEGER,0,None,0
4,4,star_rating,REAL,0,None,0
5,5,num_ratings,INTEGER,0,None,0
6,6,isbn_clean,TEXT,0,None,0



AUTHORS


,cid,name,type,notnull,dflt_value,pk
0,0,author_id,INTEGER,0,None,1
1,1,author_name,TEXT,1,None,0



BOOK_AUTHORS


...

,cid,name,type,notnull,dflt_value,pk
0,0,book_id,INTEGER,1,None,1
1,1,genre_id,INTEGER,1,None,2



SOURCE_GENRES


,cid,name,type,notnull,dflt_value,pk
0,0,source_genre_id,INTEGER,0,None,1
1,1,source_genre,TEXT,1,None,0



BOOK_SOURCE_GENRES


,cid,name,type,notnull,dflt_value,pk
0,0,book_id,INTEGER,1,None,1
1,1,source_genre_id,INTEGER,1,None,2


### 23. Test Relational Query

A sample relational query is executed to confirm that the primary `BOOKS`, `BOOK_AUTHORS`, and `AUTHORS` tables can be successfully joined.

The query retrieves highly rated books along with their authors, publication years, and Goodreads ratings. This serves as a final test that the primary keys and foreign-key relationships are functioning correctly and that the database is ready for the SQL analysis phase.

In [ ]:
query = """
SELECT
    b.name AS book_title,
    a.author_name,
    b.pub_year,
    b.star_rating
FROM books AS b
JOIN book_authors AS ba
    ON b.book_id = ba.book_id
JOIN authors AS a
    ON ba.author_id = a.author_id
ORDER BY b.star_rating DESC
LIMIT 20;
"""

top_books = pd.read_sql_query(query, conn)

display(top_books)

,book_title,author_name,pub_year,star_rating
0,Journey Through Chaos: The Valley,Ward Williams,2015,5.0
1,In The Land Of Scarabs,Janna Yeshanova,2014,5.0
2,E'S 6,Satol Yuiga,2000,5.0
3,Eternal Requiem,J.S. Chancellor,2012,5.0
4,The Fairy Godmother Dilemma: Trollspell,Danyelle Leafty,2015,5.0
5,Political Punch: Contemporary Poems on the Pol...,Fox Frazier-Foley,2016,5.0
6,Boobs In Paradise,John David Lionel Brooke,2015,5.0
7,North of Khyber,Robert E. Howard,1987,5.0
8,"Labrador Wilderness, Newfoundland and Labrador...",Llewelyn Pritchard,2011,5.0
9,"Port Hope Simpson Mysteries Vol. 2, Newfoundla...",Llewelyn Pritchard,2011,5.0


### 24. Test Book-Genre Relationship

A relational query is used to verify the connection between the `BOOKS`, `BOOK_GENRES`, and `GENRES` tables.

The query retrieves highly rated books along with their associated genres, publication years, and Goodreads ratings. This confirms that the book-to-genre many-to-many relationship is functioning correctly within the database.

In [ ]:
query = """
SELECT
    b.name AS book_title,
    g.genre_name,
    b.pub_year,
    b.star_rating
FROM books AS b
JOIN book_genres AS bg
    ON b.book_id = bg.book_id
JOIN genres AS g
    ON bg.genre_id = g.genre_id
ORDER BY b.star_rating DESC
LIMIT 20;
"""

book_genres_test = pd.read_sql_query(query, conn)

display(book_genres_test)

,book_title,genre_name,pub_year,star_rating
0,Journey Through Chaos: The Valley,action,2015,5.0
1,In The Land Of Scarabs,contemporary romance,2014,5.0
2,In The Land Of Scarabs,action,2014,5.0
3,In The Land Of Scarabs,womens fiction,2014,5.0
4,E'S 6,manga,2000,5.0
5,E'S 6,action,2000,5.0
6,Eternal Requiem,adult,2012,5.0
7,Eternal Requiem,fantasy,2012,5.0
8,The Fairy Godmother Dilemma: Trollspell,adult,2015,5.0
9,Political Punch: Contemporary Poems on the Pol...,poetry,2016,5.0


### 25. Close Database Connection

The SQLite database connection is closed after the database has been created, populated, validated, and tested.

Closing the connection ensures that all database resources are released properly. The final database location is also displayed for reference.

In [ ]:
conn.close()

print("SQLite database connection closed.")
print(f"Database created at: {db_path.resolve()}")

SQLite database connection closed.
Database created at: C:\Users\sarah\Projects\Book_Genre_Trends_Analysis\Data\goodreads_capstone.db
